In [1]:
import os
import csv
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

# ---------------- Configuration ----------------
ROOT_DIR = "opus-100-corpus/v1.0/supervised/"
SPLITS = ["train", "dev"]
NGRAMS = [1, 3, 5]
MIN_FREQ = 10

# raw text storage
raw_lines = {split: [] for split in SPLITS}

# final ngram tables
tables = {split: {n: Counter() for n in NGRAMS} for split in SPLITS}

# ---------------- Pass 1: Collect text ----------------
folders = [f for f in os.listdir(ROOT_DIR) if os.path.isdir(os.path.join(ROOT_DIR, f))]

for folder in folders:
    folder_path = os.path.join(ROOT_DIR, folder)
    lang1, lang2 = folder.split("-")
    target_lang = lang2 if lang1 == "en" else lang1 

    for filename in os.listdir(folder_path):
        filepath = os.path.join(folder_path, filename)
        if not os.path.isfile(filepath):
            continue

        split = None
        for s in SPLITS:
            if f"-{s}." in filename:
                split = s
                break
        if split is None:
            continue

        file_lang = filename.split(".")[-1]
        if file_lang != target_lang:
            continue

        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip().lower()
                if line:
                    raw_lines[split].append(line)

# ---------------- Pass 2: Build unigram vocab ----------------
char_counts = Counter()
for split in SPLITS:
    for line in raw_lines[split]:
        char_counts.update(line)

vocab = {c for c, cnt in char_counts.items() if cnt >= MIN_FREQ}

UNK_CHAR = "\uE000"
def replace_unk(line):
    return "".join(c if c in vocab else UNK_CHAR for c in line)

# ---------------- Pass 3: N-gram extraction ----------------
for split in SPLITS:
    processed = [replace_unk(line) for line in raw_lines[split]]

    for n in NGRAMS:
        padding = "<sos>" * (n - 1) if n > 1 else ""
        padded_lines = [padding + line for line in processed]

        vectorizer = CountVectorizer(
            analyzer="char",
            ngram_range=(n, n),
            lowercase=False
        )
        X = vectorizer.fit_transform(padded_lines)
        counts = X.sum(axis=0).A1
        vocab_ngrams = vectorizer.get_feature_names_out()

        for gram, count in zip(vocab_ngrams, counts):
            tables[split][n][gram] += int(count)

# ---------------- Write CSVs ----------------
def write_csv(split, n, table):
    name = {1: "unigrams", 3: "trigrams", 5: "5grams"}[n]
    filename = f"{name}_{split}.csv"

    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["char_ngram", "count"])
        for gram, count in table.items():
            writer.writerow([gram, count])

    print(f"Saved {filename}")

for split in SPLITS:
    for n in NGRAMS:
        write_csv(split, n, tables[split][n])


Saved unigrams_train.csv
Saved trigrams_train.csv
Saved 5grams_train.csv
Saved unigrams_dev.csv
Saved trigrams_dev.csv
Saved 5grams_dev.csv


In [3]:
def get_unigram_count(csv_file, target_char):
    with open(csv_file, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["char_ngram"] == target_char:
                return int(row["count"])
    return 0  # not found

# example usage
UNK_CHAR = "\uE000"
count = get_unigram_count("unigrams_train.csv", UNK_CHAR)
print(f"Count for UNK char: {count}")

Count for UNK char: 14121


In [1]:
import pandas as pd

In [11]:
import csv
import os

INPUT_FILE = "work/hashtable_counts/5grams_train.csv"
OUTPUT_PREFIX = "5grams_train_part"
ROWS_PER_FILE = 2_000_000  # adjust if needed

os.makedirs("work/hashtable_counts/split_5grams_train", exist_ok=True)

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)

    file_idx = 0
    row_count = 0
    out_file = None
    writer = None

    for row in reader:
        if row_count % ROWS_PER_FILE == 0:
            if out_file:
                out_file.close()

            out_path = f"work/hashtable_counts/split_5grams_train/{OUTPUT_PREFIX}_{file_idx}.csv"
            out_file = open(out_path, "w", newline="", encoding="utf-8")
            writer = csv.writer(out_file)
            writer.writerow(header)

            file_idx += 1

        writer.writerow(row)
        row_count += 1

    if out_file:
        out_file.close()

print(f"Split into {file_idx} files.")

Split into 29 files.
